In [1]:
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import numpy as np
from gensim.models import Word2Vec
import fasttext.util
import fasttext
import fasttext.util
import gzip
import os
import pandas as pd
import anthropic
from typing import List, Dict, Any
import json
import asyncio
import re
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
from datetime import datetime
import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # FutureWarning 제거

In [27]:
pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_excel('../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../data/info.csv')
api_key = api.loc[0][1]

In [5]:
df = df[['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', 'End feel', '치료계획',
       'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견']]

In [14]:
df.sample(10).PI

8745                                   * #47 씹으실때 통증, MOB+
24719                           우측 구치부 백반증전체적으로 금가 있고 갈려있음
8910                                                   NaN
13521    * 턱 떨림 있음* both) 3 역할 못함12345678 12345678Dr.남윤...
6637                                            치아마모,(구치부)
26549                                              * 반대 교합
22435                                                  NaN
1950                                                *#22변색
9682                                 * both) 3 교모심함* CA 많음
17636                                                  NaN
Name: PI, dtype: object

In [ ]:
import os
import json
import logging
from datetime import datetime
from typing import List, Dict, Optional
import asyncio
import pandas as pd
import anthropic
from tenacity import retry, stop_after_attempt, wait_exponential
import re
from tqdm.asyncio import tqdm as tqdm_asyncio
import nest_asyncio
nest_asyncio.apply()  # Apply the patch for nested event loops


# Disable SettingWithCopyWarning
pd.options.mode.chained_assignment = None
# processed_df = asyncio.run(process_medical_data(df_sample, api_key))
# Configuration class for better organization
class Config:
    # MODEL_NAME = "claude-3-5-haiku-20241022"
    MODEL_NAME = "claude-3-sonnet-20240229"
    MAX_TOKENS = 4096
    TEMPERATURE = 0
    BATCH_SIZE = 50
    SEMAPHORE_LIMIT = 5
    MAX_RETRIES = 3
    CHECKPOINT_DIR = "checkpoints"
    LOG_FILE = "medical_classifier.log"

# Improved logging setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(Config.LOG_FILE),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class CheckpointManager:
    """Checkpoint management class"""
    def __init__(self, checkpoint_dir: str = Config.CHECKPOINT_DIR):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(self.checkpoint_dir, exist_ok=True)
    
    def get_checkpoint_path(self, column: str) -> str:
        safe_column = column.replace("/", "_").replace("\\", "_")
        return os.path.join(self.checkpoint_dir, f"{safe_column}_checkpoint.parquet")
        # return os.path.join(self.checkpoint_dir, f"{column}_checkpoint.parquet")

    def save_checkpoint(self, df: pd.DataFrame, column: str) -> None:
        try:
            df.to_parquet(self.get_checkpoint_path(column))
            logger.info(f"Checkpoint saved for column {column}")
        except Exception as e:
            logger.error(f"Failed to save checkpoint for {column}: {str(e)}")

    def load_checkpoint(self, column: str) -> Optional[pd.DataFrame]:
        path = self.get_checkpoint_path(column)
        if os.path.exists(path):
            try:
                return pd.read_parquet(path)
            except Exception as e:
                logger.error(f"Failed to load checkpoint for {column}: {str(e)}")
        return None
        

class MedicalTextClassifier:
    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.semaphore = asyncio.Semaphore(Config.SEMAPHORE_LIMIT)
        self.checkpoint = CheckpointManager()
        self.classifiers = {
            # 'CC': self._classify_cc,
            # '약': self._classify_medication,
            # '장치': self._classify_device,
            # '습관': self._classify_habit,
            # '찜질': self._classify_hot_pack,
            # '마사지, 스트레칭': self._classify_massage,
            'PI': self._classify_present_illness,
            # 'CMO': self._classify_cmo,
            # 'MMO': self._classify_mmo,
            # 'Cap.pal': self._classify_cap_pal,
            # 'M.pal': self._classify_m_pal,
            # 'Noise': self._classify_noise,
            # 'Occlusion': self._classify_occlusion,
            # 'OJ/OB': self._classify_oj_ob,
            # 'Midline Shift': self._classify_midline_shift,
            'Deviation': self._classify_deviation,
            # 'CR-CO': self._classify_cr_co,
            # 'Tongue ridging': self._classify_tongue_ridging,
            # 'Mucosal ridging': self._classify_mucosal_ridging,
            # 'Rt': self._classify_rt,
            # 'Lt': self._classify_lt,
            # 'End feel': self._classify_end_feel,
            # '치료계획': self._classify_treatment_plan
        }

    async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Process all columns with improved error handling and checkpointing"""
        for column in self.classifiers.keys():
            if column in df.columns:
                logger.info(f"Processing column: {column}")
                df = await self._process_column_with_checkpoint(df, column)
        return df

    async def _process_column_with_checkpoint(self, df: pd.DataFrame, column: str) -> pd.DataFrame:
        """Process column with checkpoint support"""
        try:
            # Check for existing checkpoint
            checkpoint_df = self.checkpoint.load_checkpoint(column)
            if checkpoint_df is not None:
                df.update(checkpoint_df)
                logger.info(f"Resumed from checkpoint for {column}")
                return df

            # Process valid texts
            mask = df[column].notna() & df[column].str.strip().astype(bool)
            if not mask.any():
                return df

            texts_with_idx = [(idx, text) for idx, text in df.loc[mask, column].items()]
            dates_with_idx = [(idx, text) for idx, text in df.loc[mask, '날짜'].items()]

            print(texts_with_idx)
            print(dates_with_idx)

            results = await self._safe_process_batches(
                texts=[text for _, text in texts_with_idx],
                original_indices=[idx for idx, _ in texts_with_idx],
                classifier=self.classifiers[column],
                column=column
            )

            if results:
                result_df = pd.DataFrame(results).set_index('index')
                for col in result_df.columns:
                    new_col = f"{column}_{col}"
                    df[new_col] = result_df[col]

            self._cleanup_checkpoint(column)
            return df

        except Exception as e:
            logger.error(f"Critical error processing {column}: {str(e)}")
            raise

    async def _safe_process_batches(self, texts: List[str], original_indices: List[int],
                                  classifier, column: str) -> List[Dict]:
        """Process batches safely with retries and checkpointing"""
        results = []
        batch_size = Config.BATCH_SIZE

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            batch_indices = original_indices[i:i+batch_size]

            try:
                batch_results = await self._process_with_retry(
                    classifier, batch_texts, batch_indices
                )
                results.extend(batch_results)

                # Save partial results
                partial_df = pd.DataFrame(batch_results).set_index('index')
                self.checkpoint.save_checkpoint(partial_df, column)

            except Exception as e:
                logger.error(f"Batch {i//batch_size} failed: {str(e)}")
                continue

        return results

    @retry(stop=stop_after_attempt(Config.MAX_RETRIES),
           wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _process_with_retry(self, classifier, batch_texts: List[str],
                                batch_indices: List[int]) -> List[Dict]:
        """Process with retry logic"""
        async with self.semaphore:
            results = await classifier(batch_texts, self.semaphore)
            return [{"index": idx, **res} for idx, res in zip(batch_indices, results)]

    def _cleanup_checkpoint(self, column: str) -> None:
        """Clean up checkpoint after successful processing"""
        try:
            checkpoint_path = self.checkpoint.get_checkpoint_path(column)
            if os.path.exists(checkpoint_path):
                os.remove(checkpoint_path)
                logger.info(f"Checkpoint cleaned up for {column}")
        except Exception as e:
            logger.error(f"Failed to cleanup checkpoint for {column}: {str(e)}")

    @retry(stop=stop_after_attempt(Config.MAX_RETRIES),
           wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _make_api_call(self, prompt: str, semaphore: asyncio.Semaphore) -> List[Dict]:
        """Improved API call with better error handling"""
        try:
            async with semaphore:
                response = await asyncio.to_thread(
                    self.client.messages.create,
                    model=Config.MODEL_NAME,
                    max_tokens=Config.MAX_TOKENS,
                    temperature=Config.TEMPERATURE,
                    system="JSON 형식으로 응답하세요.",
                    messages=[{"role": "user", "content": prompt}]
                )

                content = response.content[0].text
                logger.debug(f"API Response: {content[:200]}...")

                result = self._validate_and_parse_json(content)
                if not result:
                    raise ValueError("Invalid JSON structure")
                return result

        except Exception as e:
            logger.error(f"API call failed: {str(e)}")
            raise

    def _validate_and_parse_json(self, content: str) -> List[Dict]:
        """Validate and parse JSON response"""
        try:
            # Extract JSON array using regex for more robust parsing
            array_pattern = r'\[(?:[^[\]]*|\[(?:[^[\]]*|\[[^[\]]*\])*\])*\]'
            matches = list(re.finditer(array_pattern, content))

            if not matches:
                return []

            longest_match = max(matches, key=lambda match: len(match.group()))
            potential_json = longest_match.group()

            parsed = json.loads(potential_json)
            if isinstance(parsed, list):
                # Iterate through each item in the list
                for item in parsed:
                    # Remove finish_reason key if present
                    item.pop('finish_reason', None)
                    # Recursively process nested objects
                    for key, value in item.items():
                        if isinstance(value, dict):  # Check if the value is a dictionary
                            value.pop('finish_reason', None)  # Remove key from nested object

                return parsed

            return []  # Return empty list if parsing fails

        except json.JSONDecodeError:
            logger.error(f"JSON parsing failed. Response content: {content[:500]}")
            return []

    # Original classifier methods remain the same but with improved error handling
    async def _classify_cc(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """Classify Chief Complaints"""
        prompt = f"""
            환자의 증상을 설문한 정보입니다. 당신은 구강내과 전문의이며, CC 설문 조사를 통해 결과적으로 입이 안벌어지거나, 나쁜 소리, 통증 등을 파악하여 환자를 분류하길 원합니다.
            다음 주요 증상(CC) 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
            
            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. location: 통증/증상 위치 (문자열)
            2. pain_type: 통증/증상 종류 (문자열)
            3. painUncomp_desc_jaw: 턱 통증/불편감 턱 관절의 통증, 소리, 움직임 제한 등과 관련된 증상을 포함하는 카테고리 (문자열)
            4. disable_desc_jaw: 턱 관절의 비정상적인 움직임, 소리, 제한된 개구 등의 증상을 다루는 카테고리 (문자열)
            5. muscle_joint_desc_stress: 스트레스로 인한 턱 근육의 긴장, 통증, 이갈이 등의 증상을 포함하는 카테고리 (문자열)
            6. dentalHistory_desc : 교정 치료, 보톡스, 물리치료 등 치과적 개입과 관련된 증상 및 경과를 다루는 카테고리 (문자열)
            7. clinic_history_desc : 턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기 등을 포함합니다. 교정 치료, 외상, 수술 등이 언급되는 카테고리 (문자열)
            8. factor_habbit : 음식 섭취 습관, 수면 자세, 이 악물기 등 일상생활과 연관된 턱 관절 증상을 포함하는 카테고리 (문자열)
            9. treat_plan : 물리치료, 약물요법, 장치치료, 보톡스 주사 등 계획된 치료 방법과 치료 경과 및 반응을 포함합니다. 치료 계획 변경, 추가 검사 등 카테고리 (문자열)
            10. severity: 통증/증상 강도 (1-5, 없으면 null)
            11. vas : vas 로 기재되어 있는 통증 점수. (int, 없으면 null)
            12. duration: 지속 기간 (명시된 경우만, 문자열)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "location": "위치",
                "pain_type": "통증 종류",
                "painUncomp_desc_jaw": "환자의 턱 통증/불편감",
                "disable_desc_jaw": "환자의 턱 관절 비정상적 움직임",
                "muscle_joint_desc_stress": "환자의 스트레스로 인한 턱 근육 긴장",
                "dentalHistory_desc": "환자의 교정 치료, 보톡스, 물리치료 등",
                "clinic_history_desc": "환자의 턱관절장애 관련 과거 병력, 치료 이력, 발병 시기 및 계기 등",
                "factor_habbit": "환자의 음식 섭취 습관, 수면 자세, 이 악물기 등",
                "treat_plan": "물리치료, 약물요법, 장치치료, 보톡스 주사 등",
                "severity": "숫자 또는 null",
                "vas": "숫자 또는 null",
                "duration": "기간 또는 null"
            }}]"""
        return await self._make_api_call(prompt, semaphore)

    async def _classify_medication(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """약물 복용 분류"""
        prompt = f"""다음 약물 복용 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. medication_type: 약물 종류 (진통제/소염제/근이완제 등)
            2. frequency: 복용 빈도 ('regular': 정기적, 'occasional': 간헐적, 'none': 미복용)
            3. duration: 복용 기간 (명시된 경우만)
            4. compliance: 복약 순응도 ('good': 양호, 'fair': 보통, 'poor': 불량)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "medication_type": "약물 종류",
                "frequency": "복용 빈도",
                "duration": "기간 또는 null",
                "compliance": "순응도"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_device(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """장치 사용 분류"""
        prompt = f"""다음 장치 사용 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. device_type: 장치 종류
            2. usage_pattern: 사용 패턴 ('constant': 상시착용, 'partial': 부분착용, 'rare': 거의미착용)
            3. duration: 사용 기간
            4. compliance: 착용 순응도 ('good': 양호, 'fair': 보통, 'poor': 불량)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "device_type": "장치 종류",
                "usage_pattern": "사용 패턴",
                "duration": "기간 또는 null",
                "compliance": "순응도"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_habit(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """습관 분류"""
        prompt = f"""다음 습관 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. habit_type: 습관 종류 (이갈이/편측성저작 등)
            2. frequency: 빈도 ('high': 매일/자주, 'medium': 가끔, 'low': 거의없음)
            3. awareness: 인지여부 ('aware': 인지, 'unaware': 미인지)
            4. improvement: 개선여부 ('improved': 개선, 'unchanged': 유지, 'worsened': 악화)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "habit_type": "습관 종류",
                "frequency": "빈도",
                "awareness": "인지여부",
                "improvement": "개선여부"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_hot_pack(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """찜질 분류"""
        prompt = f"""다음 찜질 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. status: 찜질 시행 여부 (0: 미시행, 1: 시행)
            2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
            3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
            4. method: 찜질 방법 ('hot': 온찜질, 'cold': 냉찜질, 'both': 둘 다, null: 불명확)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "status": 0 또는 1,
                "frequency": "빈도",
                "duration": 숫자 또는 null,
                "method": "방법"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_massage(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """마사지/스트레칭 분류"""
        prompt = f"""다음 마사지/스트레칭 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. type: 종류 ('massage': 마사지, 'stretching': 스트레칭, 'both': 둘다)
            2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
            3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
            4. method: 방법 ('self': 자가, 'professional': 전문가, 'both': 둘다)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "type": "종류",
                "frequency": "빈도",
                "duration": 숫자 또는 null,
                "method": "방법"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    # async def _classify_present_illness(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
    #     """현재 질환(PI) 분류 - 수정된 프롬프트"""
    #     prompt = f"""다음 현재 질환(PI) 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
    #         텍스트 목록:
    #         {texts}

    #         각 텍스트에 대해 아래 정보를 추출해주세요:
    #         - onset: 발현 시기 (예: "3개월 전", "2023년 1월")
    #         - pattern: 증상 양상 ("constant", "intermittent", "progressive")
    #         - aggravating_factors: 악화 요인 목록 (리스트 형식)
    #         - status: 현재 상태 ("improving", "unchanged", "worsening")
    #         - TMJ_PI_desc: 진단 및 검사 항목 (리스트 형식)
    #         - TMJ_PI_treatment: 물리치료 항목 (리스트 형식)
    #         - drug_treatment: 약물치료 항목 (리스트 형식)
    #         - closing_dentalgear_desc: 교합치료 항목 (리스트 형식)
    #         - PI_check: 경과관찰 항목 (리스트 형식)
    #         - PI_diagnosis_jojint: 파노라마, CT 촬영, 측두하악장애 분석검사 등을 통해 턱관절 질환을 진단하는 내용이 포함됩니다. 주로 턱관절의 퇴행성 관절염(K07.65) 진단이 많이 언급된 카테고리. (문자열)
    #         - physical_therapy : 저작근 장애(K07.66) 등을 치료하기 위해 다양한 물리치료와 자극요법이 시행되었습니다. 측두하악관절 단순/전기/복합 자극요법, 분사신장치료 등이 자주 언급된 카테고리 (문자열)
    #         - occlusal_treatment : 턱관절 질환 및 치아 교모를 해결하기 위해 교합안정장치(Splint) 치료가 많이 이루어졌습니다. APS, SS 등의 장치 종류가 언급되었고, 장치 제작을 위한 인상채득과 장치 장착 및 조정이 언급된 카테고리 (문자열)
    #         - medication_prescription : 턱관절 및 저작근 질환의 통증 조절과 치료를 위해 페리슨, 세크로, 리보트릴, 소론도 등 다양한 약물이 처방이 포함된 카테고리 (문자열)
    #         - other_treatment : 보톡스 시술, 악관절 강세척술, 고착해소술, 교합조정, 발치 등 턱관절 질환 및 교합 문제 해결을 위한 다양한 치료가 시행되었습니다. 또한 치아 마모, 파절 등의 문제를 해결하기 위한 보철 치료도 일부 언급된 카테고리 (문자열)

    #         아래 JSON 형식으로 응답해주세요:
    #         [{{
    #             "onset": "", 
    #             "pattern": "", 
    #             "aggravating_factors": [], 
    #             "status": "", 
    #             "TMJ_PI_desc": [], 
    #             "TMJ_PI_treatment": [], 
    #             "drug_treatment": [], 
    #             "closing_dentalgear_desc": [], 
    #             "PI_check": [],
    #             "PI_diagnosis_jojint": "",
    #             "physical_therapy": "",
    #             "occlusal_treatment": "",
    #             "medication_prescription": "",
    #             "other_treatment": ""
    #             }}]
    #         """
    #     return await self._make_api_call(prompt, semaphore)
# ---


    async def _classify_present_illness(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """현재 질환(PI) 분류 - 수정된 프롬프트"""

        prompt = f"""
                    다음은 턱관절장애(TMJ) 및 저작근 장애에 대한 진료 기록(PI 텍스트)입니다. 
                    문서에는 K07.65(퇴행성 관절염), K07.66(저작근의 장애), K07.63(턱관절 통증) 등의 진단 코드, 
                    파노라마/CT 촬영, 측두하악장애분석검사, 물리치료, 약물 처방, 경과 관찰 등의 내용이 포함될 수 있습니다.

                    아래 텍스트를 분석하여, 다음 14개 필드를 JSON 형식으로 추출해주세요:

                    1. onset (발현 시기)  
                    - 예: "3개월 전", "2023년 1월", "발병 시기 미상"  
                    - 텍스트에서 구체적으로 언급된 경우만 추출하고, 없으면 `""`(빈 문자열)

                    2. pattern (증상 양상)  
                    - "constant" (지속성), "intermittent" (간헐성), "progressive" (점진적 악화)  
                    - 명확히 언급된 경우에만 지정. 없으면 `""`

                    3. aggravating_factors (악화 요인 목록, 배열)  
                    - 예: ["딱딱한 음식", "스트레스", "이 악물기"]  
                    - 여러 개라면 배열에 순서대로 담고, 없으면 `[]`

                    4. status (현재 상태)  
                    - "improving" (호전), "unchanged" (변화 없음), "worsening" (악화)  
                    - 명시되지 않으면 `""`

                    5. TMJ_PI_desc (진단/검사 항목, 배열)  
                    - 파노라마, CT, 측두하악장애분석검사, 초음파, T-scan 등 실시된 검사 이름을 배열로 적습니다.  
                    - 예: ["파노라마", "Cone Beam CT"]

                    6. TMJ_PI_treatment (물리치료 항목, 배열)  
                    - 분사신장치료, 전기자극치료, 복합자극치료, 물리치료 등  
                    - 예: ["측두하악관절자극요법-단순", "분사신장치료"]

                    7. drug_treatment (약물치료 항목, 배열)  
                    - 예: ["소론도정(프레드니솔론)", "페리슨정(에페리손염산염)"]  
                    - 복용 방법, 용량, 횟수 등은 이 필드가 아닌 `medication_prescription`에 기입

                    8. closing_dentalgear_desc (교합치료 항목, 배열)  
                    - 교합안정장치(Splint), 교합조정, 보톡스(교합개선 목적) 등 교합 관련 치료가 있으면 적습니다.  
                    - 없으면 `[]`

                    9. PI_check (경과관찰 항목, 배열)  
                    - “증상 체크 [2주후]”, “1개월 후 재내원”, “재평가 예정” 등 주기적 검진/재평가 계획  
                    - 예: ["증상 ck [2주후]"]

                    10. PI_diagnosis_jojint (턱관절 진단 내용, 문자열)  
                    - K07.65 퇴행성 관절염, K07.63 턱관절 통증, K07.66 저작근 장애 등 진단명  
                    - 예: "퇴행성 관절염 (K07.65)"

                    11. physical_therapy (저작근 장애(K07.66) 물리치료 등, 문자열)  
                    - 저작근 장애를 치료하기 위해 시행된 물리치료나 자극요법(단순/전기/복합), 분사신장치료 등의 종합 요약  
                    - 예: "측두하악관절 단순/전기/복합 자극, 분사신장치료"

                    12. occlusal_treatment (교합안정장치(Splint) 등 장치치료, 문자열)  
                    - 예: "APS, SS 장치 장착 및 조정"  
                    - 교합 조정, 장치 제작 등

                    13. medication_prescription (약물 처방 상세, 문자열)  
                    - “페리슨정(에페리손염산염) 1/1회/14일 :: 취침 직전 복용” 처럼, 복용 용법/횟수/기간 등을 구체적으로 적습니다.  
                    - 여러 개라면 문장으로 나열  
                    - 예: "페리슨정(에페리손염산염) - 1/1회/14일, 소론도정(프레드니솔론) - 1/1회/14일"

                    14. other_treatment (그 외 보톡스 시술, 악관절 강세척술, 교합조정, 발치 등, 문자열)  
                    - “보톡스 시술”, “악관절 세척술” 등  
                    - 텍스트에서 확인되면 구체적으로 작성, 없으면 `""`

                    ---

                    ### [중요] 구체적 예시(Few-shot) 출력

                    아래는 실제 텍스트 일부를 예시로 들어, 모델이 어떤 식으로 JSON을 구성해야 하는지 보여주는 예시입니다.  
                    모델은 이 예시 패턴을 학습하여, 유사한 구조로 응답하면 됩니다.

                    #### 예시 입력:



                    텍스트 목록:
                    {texts}

                    JSON 예시:
                    [
                    {{
                        [{
                        "onset": "",
                        "pattern": "",
                        "aggravating_factors": [],
                        "status": "",
                        "TMJ_PI_desc": [
                            "파노라마(특수)",
                            "Cone Beam CT",
                            "측두하악장애분석검사(초음파)"
                        ],
                        "TMJ_PI_treatment": [
                            "TMJ자극요법-단순",
                            "분사신장치료",
                            "복합자극치료"
                        ],
                        "drug_treatment": [
                            "페리슨정(에페리손염산염)",
                            "소론도정(프레드니솔론)"
                        ],
                        "closing_dentalgear_desc": [],
                        "PI_check": [
                            "증상 ck [2주후]"
                        ],
                        "PI_diagnosis_jojint": "퇴행성 관절염 (K07.65)",
                        "physical_therapy": "저작근 장애(K07.66) 자극요법(단순/복합), 분사신장치료",
                        "occlusal_treatment": "",
                        "medication_prescription": "페리슨정(에페리손염산염) - 1/1회/14일 (취침 전), 소론도정(프레드니솔론) - 1/1회/14일 (아침)",
                        "other_treatment": ""
                        }]
                    }}
                    ]
                """
        return await self._make_api_call(prompt, semaphore)

    # async def _classify_cmo(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
    #     """
    #     CMO 컬럼: mm_after_treatment, mm_before_treatment, pain_popping_etc, treatment_method
    #     """
        
    #     return await self._make_api_call(prompt, semaphore)

    async def _classify_mmo(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        MMO 컬럼: 초기 개구량, 처치 후 개구량, 통증 여부, 좌우 구분, 기타 증상
        """
        prompt = f"""
            다음 MMO 관련 텍스트를 분석하여 JSON 형식으로 분류해주세요.
            각 텍스트에 대해 아래 정보를 추출:
            1. initial_opening: 초기 개구량 (숫자, 없으면 null)
            2. post_treatment_opening: 처치 후 개구량 (숫자, 없으면 null)
            3. pain: 통증 여부 ("pain"/"no pain"/"unknown")
            4. side: 좌우 구분 ("Lt"/"Rt"/"both"/"unknown")
            5. other_symptoms: 기타 증상 (문자열)

            텍스트 목록:
            {texts}

            JSON 예시:
            [
            {{
                "initial_opening": 30,
                "post_treatment_opening": 40,
                "pain": "pain",
                "side": "Lt",
                "other_symptoms": "걸림(catch) 있음"
            }}
            ]
        """
        return await self._make_api_call(prompt, semaphore)

    async def _classify_cap_pal(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        Cap.pal 컬럼
        - Negative
        - Unilateral Positive
        - Bilateral Positive
        - Severity
        - Unspecified
        """
        prompt = f"""
        다음 Cap.pal 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. cap_pal_result: ("Negative", "Unilateral Positive", "Bilateral Positive", "Severity", "Unspecified")
        2. detail: 추가 설명 (문자열, 예: 'Lt > Rt', '턱떨림 심함' 등)

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "cap_pal_result": "Unilateral Positive",
            "detail": "Rt)+"
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)

    async def _classify_m_pal(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        M.pal 컬럼
        - Masseter 통증, Temporalis 통증, 압통점(TP), SCM 연관통, 기타 소견
        """
        prompt = f"""
        다음 M.pal 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. masseter_pain: 교근(Masseter) 통증 정도 (문자열)
        2. temporalis_pain: 측두근(Temporalis) 통증 정도 (문자열)
        3. tender_point: 압통점 유무/부위 (문자열)
        4. scm_referred_pain: SCM 연관통 여부 (문자열)
        5. other_findings: 기타 소견 (문자열)

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "masseter_pain": "Lt +",
            "temporalis_pain": "Rt +/-",
            "tender_point": "both sides",
            "scm_referred_pain": "no",
            "other_findings": "클릭음 변화"
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)

    async def _classify_noise(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        Noise 컬럼
        - Click, Popping, Crepitus, None
        """
        prompt = f"""
        다음 관절음(Noise) 관련 텍스트를 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. noise_type: ("Click", "Popping", "Crepitus", "None", "unknown")
        2. noise_position: (문자열, 예: Lt, Rt),
        3. noise_amount: (정수, 0~5로 범주화 예: 심함은 5),
        

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "noise_type": "Click",
            "noise_position": "Lt",
            "noise_amount":5
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)


    async def _classify_occlusion(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        Occlusion 컬럼
        - 정상교합, 부분교합, 비접촉, 마모/상실, 기타
        """
        prompt = f"""
        다음 교합(Occlusion) 관련 텍스트를 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. occlusion_type: ("정상교합", "부분교합", "비접촉", "마모/상실", "기타")
        2. detail: 구체 설명 (문자열)

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "occlusion_type": "부분교합",
            "detail": "Rt)567 Lt)4567"
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)

    async def _classify_oj_ob(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        OJ/OB 컬럼
        - 정상 범위, 과도한 수평피개, 과도한 수직피개, 전치부 개방교합, 측정 불가
        """
        prompt = f"""
        OB는 Overbite는 상악(윗니)의 앞니가 하악(아랫니)의 앞니를 덮는 정도를 나타냄 (정상적인 오버바이트는 약 2~4mm 정도가 적당)
        OJ는 Overjet는 윗니가 아랫니보다 얼마나 튀어나와 있는지를 측정하는 것 (정상적인 오버젯은 보통 2~3mm)
        OB = 2mm 보다 작으면 안좋음. 마이너스가 안좋은 것임.
        OJ = 3mm 이상 커지만 안좋음.
        
        정상 범위	OJ와 OB가 2-4mm 사이로 정상 범위에 속하는 경우
        과도한 수평피개	OJ가 4mm 이상으로 수평적 과도한 피개를 보이는 경우
        과도한 수직피개	OB가 음수값을 가지거나 4mm 이상으로 수직적 과도한 피개를 보이는 경우
        전치부 개방교합	OJ 또는 OB가 0 또는 음수값을 가져 전치부 개방교합을 보이는 경우
        측정 불가	OJ/OB 값이 기록되지 않았거나 측정이 어려운 경우
        
        다음 OJ/OB 관련 텍스트를 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. OJ : '/' 앞의 숫자를 추출한 Int 형식의 숫자
        2. OB : '/' 뒤의 숫자를 추출한 Int 형식의 숫자
        1. oj_ob_category: ("정상 범위", "과도한 수평피개", "과도한 수직피개", "전치부 개방교합", "측정 불가")

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "OJ": "4",
            "OB": "2",
            "oj_ob_category : "과도한 수평피개"
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)


    async def _classify_midline_shift(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        Midline Shift 컬럼
        - 상악 편위, 하악 편위, 편위 없음, 측정 불가
        """
        prompt = f"""
        다음 Midline Shift 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. shift_type: ("상악 편위", "하악 편위", "편위 없음", "측정 불가")
        2. mds_direction: 편위 방향 (문자열, 예 : LT, RT, both)
        3. mds_amount: 편위 정도 (정수, 예: 2)
        

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "shift_type": "상악 편위",
            "mds_direction": "LT"
            "mds_amount" : 2
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)

    async def _classify_deviation(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        Deviation 컬럼
        - Left, Right, Straight, Mixed, Unspecified
        """
        prompt = f"""
        입을 벌리면서 위  턱에 대해 아래 턱이 어떻게 움직이는지를 포함하는 열입니다.
        L이 가장 안좋으며 S는 나쁘지 않습니다. 
        
        다음 Deviation 관련 텍스트를 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. deviation_type: (문자열, 예:"L","S" )
        2. deviation_direcion: (문자열, 예: "왼쪽")
        3. deviation_power: (0~3 사이의 정수, 0은 없음, 3은 심함)

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "deviation_type": "Left",
            "deviation_direcion": "약간의 편위"
            "deviation_power": 1
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)

    async def _classify_cr_co(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        CR-CO 컬럼
        - 없음, 1mm 이하, 1-2mm, 2mm 초과, 방향 표시
        """
        prompt = f"""
         Centric Relation
            - Centric Relation은 교합이 가장 안정적이고 균형 잡힌 상태일 때, 즉 두 턱이 제대로 맞물리는 위치를 나타냅니다.
            - CR은 근육과 인대가 최대로 긴장되거나 최적의 위치에 있을 때로, 이 상태에서 하악을 상악과 맞추는 것이 중요합니다.
        - Centric Occlusion
            - 실제로 두 턱이 닫힐 때, 즉 치아가 맞물리는 상태를 말합니다. 하악의 치아가 상악의 치아와 접촉하는 지점으로, Centric Occlusion은 교합의 "물어보는" 상태를 의미합니다.
            - CO는 일반적으로 CR과 일치하는 것이 이상적이나, 때로는 CR과 CO가 일치하지 않는 경우도 있을 수 있습니다. 이런 경우에는 교정치료가 필요할 수 있습니다.
        
        다음 CR-CO 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. cr_co_direction : (문자열, 예: "오른쪽 뒤로", "오른쪽")
        2. cr_co_amount: (정수, 예: 1, 없을 시 0)

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "cr_co_direction": "오른쪽",
            "cr_co_amount": "1"
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)

    async def _classify_tongue_ridging(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        Tongue ridging 컬럼
        - Positive, Negative, Not specified, Mild
        """
        prompt = f"""
        강한 양성 (++)	혀의 융기가 매우 뚜렷하고 심각한 수준으로 나타나는 경우
        약한 양성 (+)	혀의 융기가 경미하게 관찰되는 상태
        음성 (-)	혀의 융기가 전혀 없거나 관찰되지 않는 상태
        미확인 (n/s)	데이터 확인이 불가능하거나 판단할 수 없는 상태        
        음성을 0으로, 강한 양성을 2로 정수 범주로 변경.
        + 혹은 약하게 있다 등의 문자열은 1, 심함은 3
        
        다음 Tongue ridging 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. tongue_ridging: (정수형, 예: 1)

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "tongue_ridging": 1
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)

    async def _classify_mucosal_ridging(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        Mucosal ridging 컬럼
        - Positive, Negative, Not specified, Indeterminate
        """
        prompt = f"""
        강한 양성 (++)	혀의 융기가 매우 뚜렷하고 심각한 수준으로 나타나는 경우
        약한 양성 (+)	혀의 융기가 경미하게 관찰되는 상태
        음성 (-)	혀의 융기가 전혀 없거나 관찰되지 않는 상태
        미확인 (n/s)	데이터 확인이 불가능하거나 판단할 수 없는 상태        
        음성을 0으로, 강한 양성을 2로 정수 범주로 변경.
        + 혹은 약하게 있다 등의 문자열은 1, 심함은 3
        
        다음 Mucosal ridging 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. mucosal_ridging: (정수형, 예: 1)

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "mucosal_ridging": 1
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)

    async def _classify_rt(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        Rt 컬럼
        - 정상, 경도 이상, 중등도 이상, 고도 이상, 측정 불가
        """
        prompt = f"""
        화살표 전후로 힘 안 줬을 때, 꽉 물었을 때의 근육 두께에 대한 정보이며 중요한 정보입니다.
        남자는 1.5, 여자는 1.3 미만으로 가는게 목표입니다.        
        
        다음 Rt 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. before_posing : (실수형, 화살표 이전의 실수. 화살표가 없다면 Null. 예 : 1.18)
        2. after_posing : (실수형, 화살표 이후의 실수. 화살표가 없다면 이 정보임. 예 : 1.63)

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "rt_before_posing": 1.2,
            "rt_after_posing": 2.1
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)

    async def _classify_lt(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        Lt 컬럼
        - 정상범위, 경도상승, 중등도상승, 고도상승, 측정누락
        """
        prompt = f"""
        화살표 전후로 힘 안 줬을 때, 꽉 물었을 때의 근육 두께에 대한 정보이며 중요한 정보입니다.
        남자는 1.5, 여자는 1.3 미만으로 가는게 목표입니다.        
        
        다음 Rt 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. before_posing : (실수형, 화살표 이전의 실수. 화살표가 없다면 Null. 예 : 1.18)
        2. after_posing : (실수형, 화살표 이후의 실수. 화살표가 없다면 이 정보임. 예 : 1.63)

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "lt_before_posing": 1.2,
            "lt_after_posing": 2.1
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)

    async def _classify_end_feel(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        End feel 컬럼
        - Soft, Hard, Intermediate, Not specified
        """
        prompt = f"""
        다음 End feel 관련 텍스트를 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        1. end_feel_type: ("Soft", "Hard", "Intermediate", "Not specified")
        2. comment: 기타 코멘트 (문자열)

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "end_feel_type": "Hard",
            "comment": "뻣뻣함"
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)

    async def _classify_treatment_plan(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """
        치료계획 컬럼
        - 물리치료, 장치 관리, 습관 조절, 약물/주사 치료, 경과 관찰
        """
        prompt = f"""
        

        다음 치료계획 관련 텍스트를 분석하여 JSON 형식으로 분류해주세요.
        각 텍스트에 대해 아래 정보를 추출:
        plat_tracking : 문자열, 환자의 향후 진료 일정, 재방문 시기, 경과 관찰 등과 관련된 계획
        plat_extra_treat : 문자열, 물리치료, 보톡스, 장치 처방 등 구체적인 치료 접근 방식에 대한 계획
        next_extra_treat : 문자열, 현재 치료 이후 필요한 추가적인 의학적 개입이나 환자의 자가관리 지침
        next_evaluate : 문자열, 초음파 검사 등 향후 필요한 진단적 평가 및 재평가 계획
        next_schedule : 일 수로 변환한 정수형, 예: 1주 후 : 7, 1달 후 : 30

        텍스트 목록:
        {texts}

        JSON 예시:
        [
          {{
            "plat_tracking":"장치 ck"
            "plat_extra_treat":"물리치료"
            "next_extra_treat":""
            "next_evaluate":""
            "next_schedule":"7"
          }}
        ]
        """
        return await self._make_api_call(prompt, semaphore)

# ---

async def process_medical_data(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """Process medical data with comprehensive error handling and logging"""
    classifier = MedicalTextClassifier(api_key)
    start_time = datetime.now()
    logger.info(f"Starting medical data processing at {start_time}")

    try:
        # Process data
        processed_df = await classifier.process_all_columns(df)

        # Log statistics
        end_time = datetime.now()
        processing_time = end_time - start_time
        total_rows = len(df)
        processed_columns = [col for col in df.columns if col in classifier.classifiers]

        logger.info("=== Processing Summary ===")
        logger.info(f"Total time: {processing_time}")
        logger.info(f"Total rows processed: {total_rows}")
        logger.info(f"Columns processed: {processed_columns}")

        # Calculate success rates for each column
        for col in processed_columns:
            total_entries = df[col].notna().sum()
            processed_entries = sum(1 for col_name in processed_df.columns
                                 if col_name.startswith(f"{col}_")
                                 and processed_df[col_name].notna().any())
            success_rate = (processed_entries / total_entries * 100) if total_entries > 0 else 0
            logger.info(f"{col} - Success rate: {success_rate:.2f}%")

        return processed_df

    except Exception as e:
        logger.critical(f"Critical error during medical data processing: {str(e)}")
        raise
    finally:
        # Cleanup (without await)
        if classifier.client:  # Check if client is initialized
            classifier.client.close() # Run close() without await
        logger.info("Processing completed and resources cleaned up")

if __name__ == "__main__":
    # Setup logging
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(Config.LOG_FILE),
            logging.StreamHandler()
        ]
    )
    logger = logging.getLogger(__name__)

    try:
        # Load sample data
        # df_sample = df.sample(10)
        df_sample = df.query("환자번호 == '2212-114'")

        # Set API key (should be in environment variable or config file in production)

        # Process data
        logger.info("Starting sample data processing")
        loop = asyncio.get_event_loop()
        processed_df = loop.run_until_complete(process_medical_data(df_sample, api_key))

        # Save results
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = f'processed_medical_data_{timestamp}.parquet'
        processed_df.to_parquet(output_file)
        logger.info(f"Data successfully saved to {output_file}")

        # Print basic statistics
        logger.info("\n=== Processing Results ===")
        for column in processed_df.columns:
            if '_' in column:  # Only show derived columns
                valid_count = processed_df[column].notna().sum()
                logger.info(f"{column}: {valid_count} valid entries")

                if processed_df[column].dtype in ['object', 'category']:
                    value_counts = processed_df[column].value_counts()
                    logger.info(f"Value distribution:\n{value_counts}\n")

    except Exception as e:
        logger.error(f"Main execution failed: {str(e)}")
        sys.exit(1)
    finally:
        logger.info("Program execution completed")



2025-02-23 23:17:43,285 - __main__ - INFO - Starting sample data processing
2025-02-23 23:17:43,293 - __main__ - INFO - Starting medical data processing at 2025-02-23 23:17:43.293330
2025-02-23 23:17:43,293 - __main__ - INFO - Processing column: PI


[(15191, '12345678 12345678Dr.남윤진료측두하악장애분석검사, 파노라마, CT촬영 - K07.65 턱관절의 퇴행성관절염- 파노라마- 파노라마(특수)- Cone Beam CT- 측두하악장애분석검사판독소견 osteophyte formation on both TMJs- Conclusion : 턱관절의 퇴행성 관절염 (K07.65)파노라마 턱관절염 통증 호소하여 파노라마 촬영함파노라마(특수) 턱관절염 통증 호소하여 악관절 파노라마 촬영함Cone Beam CT 퇴행성 관절염 의심으로 CT촬영함측두하악장애분석검사 측두하악장애분석검사 진행- 초음파12345678 12345678Dr.남윤진료TMJ자극요법-단순, 분사신장 - K07.66 저작근의 장애- 분사신장치료- 측두하악관절자극요법-단순자극12345678 12345678Dr.남윤진료TMJ자극요법-전기 - K07.63 달리 분류되지 않은 턱관절의 통증- 측두하악관절자극요법-전기자극12345678 12345678Dr.남윤진료TMJ자극요법-복합 - K07.63 달리 분류되지 않은 턱관절의 통증- 측두하악관절자극요법-복합자극- 페리슨정(에페리손염산염) - 1/1회/14일 :: 취침 직전 복용- 소론도정(프레드니솔론) - 1/1회/14일 :: 아침 기상 직후 복용물리치료 , 증상 ck [2주후]'), (15195, '12345678 12345678Dr.남윤진료TMJ자극요법-단순 - K07.66 저작근의 장애- 측두하악관절자극요법-단순자극- 분사신장치료측두하악관절자극요법-단순자극 12월 9일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료TMJ자극요법-전기 - K07.66 저작근의 장애- 측두하악관절자극요법-전기자극측두하악관절자극요법-전기자극 12월 9일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료TMJ자극요법-복합 - K07.66 저작근의 장애- 측두하악관절자극요법-복합자극측두하악관절자극요법-복합자극 12월 9일에 측두하악장애 분석검사 시행1234567Dr.남윤

2025-02-23 23:17:57,020 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-02-23 23:17:57,025 - __main__ - INFO - Checkpoint saved for column PI
2025-02-23 23:17:57,029 - __main__ - INFO - Checkpoint cleaned up for PI
2025-02-23 23:17:57,030 - __main__ - INFO - Processing column: Deviation
2025-02-23 23:17:57,031 - __main__ - INFO - === Processing Summary ===
2025-02-23 23:17:57,032 - __main__ - INFO - Total time: 0:00:13.738275
2025-02-23 23:17:57,032 - __main__ - INFO - Total rows processed: 6
2025-02-23 23:17:57,032 - __main__ - INFO - Columns processed: ['PI', 'Deviation']
2025-02-23 23:17:57,036 - __main__ - INFO - PI - Success rate: 166.67%
2025-02-23 23:17:57,036 - __main__ - INFO - Deviation - Success rate: 0.00%
2025-02-23 23:17:57,038 - __main__ - INFO - Processing completed and resources cleaned up
2025-02-23 23:17:57,052 - __main__ - INFO - Data successfully saved to processed_medical_data_20250223_231757.parquet
2025-02-23 23:1

In [106]:
processed_df[['환자번호', '날짜','CC',
       'CC_location','CC_pain_type', 'CC_painUncomp_desc_jaw', 'CC_disable_desc_jaw',
       'CC_muscle_joint_desc_stress', 'CC_dentalHistory_desc',
       'CC_factor_habbit', 'CC_severity', 'CC_duration'
        ]]

,환자번호,날짜,CC,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_factor_habbit,CC_severity,CC_duration
15191,2212-114,2022-12-23,당일미예약)턱관절 소리 및 통증6-7개월전부터입을 크게 벌릴때 오른쪽 딱딱 소리와 ...,오른쪽 턱,"통증, 딱딱 소리",입을 크게 벌릴 때 오른쪽 턱에서 딱딱 소리가 나고 통증이 있음,입을 크게 벌리면 오른쪽 턱이 틀어져 벌어짐,스트레스는 없다고 함,없음,"딱딱하고 질긴 음식은 잘 먹지 않음, 이갈이는 모르겠다고 함, 유도 운동(취미)을 ...",2,6-7개월
15192,2212-114,2023-02-18,"내과 #2물리치료 , 증상 ck증상: 입 벌릴때 오른쪽 딱딱소리와 턱이 틀어져서 벌...",오른쪽 턱,"통증, 딱딱 소리",입을 벌릴 때 오른쪽 턱에서 딱딱 소리가 나고 통증이 있음,입을 벌리면 오른쪽 턱이 틀어져 벌어짐,없음,보톡스 주사 치료 계획,없음,2,None
15193,2212-114,2023-03-20,"구강내과#2처음갔던 것처럼 다시 아파요, 증상 ck-> 보톡스 주사맞는 것만 해볼게...",턱,"통증, 딱딱 소리",입을 다물 때 딱딱 소리가 나고 통증이 있음,턱이 틀어지면서 벌어짐,없음,보톡스 주사 치료,치아 끼리 닿지 않도록 신경 쓰지 않음,4,None
15194,2212-114,2023-04-06,"구강내과#3물리치료 , 증상 ck / 근육두께ck증상: 입 다물떄 딱딱 소리나고 통...",턱,"통증, 딱딱 소리",입을 다물 때 딱딱 소리가 나고 통증이 있음,없음,없음,물리치료 받음,없음,3,None
15195,2212-114,2023-04-21,"구강내과#4물리치료 , 증상 ck증상: 입 다물때 딱딱 소리 동일해요 ...",오른쪽 턱,"통증, 딱딱 소리",입을 다물 때 딱딱 소리가 남,입을 벌릴 때 오른쪽 턱 통증이 있음,없음,물리치료 받음,없음,3,None
15196,2212-114,2023-05-10,"구강내과#5물리치료 , 증상 ck APS 어머님께 원장님 재상담보호자 동반증상: 입...",오른쪽 턱,"통증, 딱딱 소리",입을 다물 때 딱딱 소리가 남,없음,없음,물리치료 받음,씹거나 입을 벌릴 때 통증이 심해짐,5,None


In [111]:
processed_df[['환자번호', '날짜','약',
       '약_medication_type',
       '약_frequency', '약_duration', '약_compliance', '습관_habit_type',
       '습관_frequency', '습관_awareness', '습관_improvement',
       # '마사지, 스트레칭_type',
       # '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration', '마사지, 스트레칭_method'
        ]]

,환자번호,날짜,약,약_medication_type,약_frequency,약_duration,약_compliance,습관_habit_type,습관_frequency,습관_awareness,습관_improvement
15191,2212-114,2022-12-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15192,2212-114,2023-02-18,약: 깜빡하고 약 하나도 안 먹었어요.,None,occasional,None,poor,편측성저작,low,aware,improved
15193,2212-114,2023-03-20,NaN,NaN,NaN,NaN,NaN,편측성저작,low,aware,unchanged
15194,2212-114,2023-04-06,NaN,NaN,NaN,NaN,NaN,편측성저작,low,aware,improved
15195,2212-114,2023-04-21,NaN,NaN,NaN,NaN,NaN,이갈이,low,aware,improved
15196,2212-114,2023-05-10,NaN,NaN,NaN,NaN,NaN,이갈이,low,aware,improved


In [112]:
processed_df[[
    'PI', 'PI_onset',
    'PI_pattern', 'PI_aggravating_factors', 'PI_status', 'PI_TMJ_PI_desc',
    'PI_TMJ_PI_treatment', 'PI_drug_treatment',
    'PI_closing_dentalgear_desc', 'PI_PI_check'
        ]]

,PI,PI_onset,PI_pattern,PI_aggravating_factors,PI_status,PI_TMJ_PI_desc,PI_TMJ_PI_treatment,PI_drug_treatment,PI_closing_dentalgear_desc,PI_PI_check
15191,"12345678 12345678Dr.남윤진료측두하악장애분석검사, 파노라마, CT촬영...",,,[],,"[파노라마, 파노라마(특수), Cone Beam CT, 측두하악장애분석검사]",[],"[페리슨정(에페리손염산염), 소론도정(프레드니솔론)]",[],"[물리치료, 증상 ck [2주후]]"
15192,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15194,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15195,12345678 12345678Dr.남윤진료TMJ자극요법-단순 - K07.66 저작...,,,[],,[],"[측두하악관절자극요법-단순자극, 분사신장치료, 측두하악관절자극요법-전기자극, 측두하...",[],[],"[물리치료, 증상 ck [2주후]]"
15196,[진행중] 상악 : APS(APS만 제작 (2월 SS 제작비용으로 안내))[예정] ...,,,[],,[],[],[],"[상악 : APS(APS만 제작 (2월 SS 제작비용으로 안내)), 하악 : SS(...",[]


In [128]:
processed_df

,환자번호,날짜,CC,약,장치,습관,찜질,"마사지, 스트레칭",PI,CMO,MMO,Cap.pal,M.pal,Noise,Loading,Occlusion,OJ/OB,Class,Midline Shift,Deviation,CR-CO,Tongue ridging,Mucosal ridging,Ultrasono,Rt,Lt,End feel,치료계획,T-scan 악화/개선,CBCT 악화/개선,CBCT 판독소견,PI_patientId,PI_doctorName,PI_diagnosis,PI_treatments,PI_notes
15191,2212-114,2022-12-23,당일미예약)턱관절 소리 및 통증6-7개월전부터입을 크게 벌릴때 오른쪽 딱딱 소리와 ...,NaN,NaN,NaN,NaN,NaN,"12345678 12345678Dr.남윤진료측두하악장애분석검사, 파노라마, CT촬영...",25mm --> mm after spray and stretch,33mm --> mm after spray and stretch,both) cap+,Rt) M Rt)M+,-,-,4567/4567,#21기준 3/2,1,하악 왼쪽 2mm,NaN,-,+,+,NaN,1.06 -> 1.58,1.10 -> 1.61,soft,"당일 내원, 진료의뢰서 지참하셔서 스캔완료/현",NaN,NaN,NaN,12345678,Dr.남윤,"[{'code': 'K07.65', 'description': '턱관절의 퇴행성관절...","[파노라마, 파노라마(특수), Cone Beam CT, 측두하악장애분석검사, 초음파]","턱관절염 통증 호소하여 파노라마 촬영함, 퇴행성 관절염 의심으로 CT촬영함, 측두하..."
15192,2212-114,2023-02-18,"내과 #2물리치료 , 증상 ck증상: 입 벌릴때 오른쪽 딱딱소리와 턱이 틀어져서 벌...",약: 깜빡하고 약 하나도 안 먹었어요.,NaN,습관: 이랑 이 안 닿게 했어요. 질기고 딱딱한 음식 안 먹었어요.,온찜질: 안했어요.,NaN,NaN,25mm --> mm after spray and stretch,33mm --> mm after spray and stretch,Rt) +,Rt) M Rt)M+,-,-,4567/4567,#21기준 3/2,1,하악 왼쪽 2mm,NaN,-,+,+,NaN,1.06 -> 1.58,1.10 -> 1.61,soft,"-> 아이 아빠랑 상의해보고 다음번에 정해서 올게요, 보톡스는 위험해서 좀 그렇다고...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15193,2212-114,2023-03-20,"구강내과#2처음갔던 것처럼 다시 아파요, 증상 ck-> 보톡스 주사맞는 것만 해볼게...",NaN,NaN,"습관: 딱딱하고 질긴 음식은 거의 안먹어요, 유도는 계속 하고 있어요.",찜질: 안했어요.,NaN,NaN,25mm --> mm after spray and stretch,33mm --> mm after spray and stretch,Rt) +,Rt) M Rt)M+,-,-,4567/4567,#21기준 3/2,1,하악 왼쪽 2mm,NaN,-,+,+,NaN,1.06 -> 1.58,1.10 -> 1.61,soft,"물리치료 , 증상 ck [1개월후]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15194,2212-114,2023-04-06,"구강내과#3물리치료 , 증상 ck / 근육두께ck증상: 입 다물떄 딱딱 소리나고 통...",NaN,NaN,"습관: 딱딱하고 질긴 음식은 안먹어요, 치아끼리 닿지 않도록 해요.",찜질: 냉찜질만 하고 안했어요.,NaN,NaN,25mm --> 40mm after spray and stretch 고착 마취,28mm --> mm after spray and stretch,Rt) +,Rt) M Rt)M+,-,-,4567/4567,#21기준 3/2,1,하악 왼쪽 2mm,NaN,-,+,+,NaN,1.06 -> 1.58/->1.32,1.10 -> 1.61/->1.37,soft,"물리치료 , 증상 ck [2주후]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15195,2212-114,2023-04-21,"구강내과#4물리치료 , 증상 ck증상: 입 다물때 딱딱 소리 동일해요 ...",NaN,NaN,습관: 질기고 딱딱 피하고 치아 물지 않으려고 했어요,찜질: 안했어요,NaN,12345678 12345678Dr.남윤진료TMJ자극요법-단순 - K07.66 저작...,30mm --> 40mm after spray and stretch 고착 마취,32mm --> mm after spray and stretch,Rt) +,Rt) M Rt)M+,-,-,4567/4567,#21기준 3/2,1,하악 왼쪽 2mm,NaN,-,+,+,NaN,1.06 -> 1.58/->1.32,1.10 -> 1.61/->1.37,soft,NaN,NaN,NaN,NaN,12345678,Dr.남윤,"[{'code': 'K07.66', 'description': '저작근의 장애'}]","[분사신장치료, 측두하악관절자극요법-단순자극]",NaN
15196,2212-114,2023-05-10,"구강내과#5물리치료 , 증상 ck APS 어머님께 원장님 재상담보호자 동반증상: 입...",NaN,NaN,습관: 질기고 딱딱 피하고 치아 닿지 않게 힘 풀려고 했어요,찜질: 안했어요.,NaN,[진행중] 상악 : APS(APS만 제작 (2월 SS 제작비용으로 안내))[예정] ...,30mm --> 40mm after spray and stretch 고착 마취,32mm --> mm after spray and stretch,Rt) +,Rt) M Rt)M+,-,-,4567/4567,#21기준 3/2,1,하악 왼쪽 2mm,NaN,-,+,+,NaN,1.06 -> 1.58/->1.32,1.10 -> 1.61/->1.37,soft,"물리치료 , APS del [2주후]",NaN,NaN,NaN,12345678,Dr.남윤,"[{'code': 'K07.63', 'description': '달리 분류되지 않은...","[측두하악관절자극요법-전기자극, 측두하악관절자극요법-복합자극, 페리슨정(에페리손염산...","물리치료, 증상 ck [2주후]"


In [116]:
print(processed_df.columns.tolist())

['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO', 'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB', 'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging', 'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', 'End feel', '치료계획', 'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견', 'CC_location', 'CC_pain_type', 'CC_painUncomp_desc_jaw', 'CC_disable_desc_jaw', 'CC_muscle_joint_desc_stress', 'CC_dentalHistory_desc', 'CC_clinic_history_desc', 'CC_factor_habbit', 'CC_treat_plan', 'CC_severity', 'CC_vas', 'CC_duration', '약_medication_type', '약_frequency', '약_duration', '약_compliance', '습관_habit_type', '습관_frequency', '습관_awareness', '습관_improvement', 'PI_onset', 'PI_pattern', 'PI_aggravating_factors', 'PI_status', 'PI_TMJ_PI_desc', 'PI_TMJ_PI_treatment', 'PI_drug_treatment', 'PI_closing_dentalgear_desc', 'PI_PI_check', 'PI_PI_diagnosis_jojint', 'PI_physical_therapy', 'PI_occlusal_treatment', 'PI_medication_prescription', 'PI_other_treatment', 'CMO_mm_after_tr

In [123]:
processed_df[['환자번호', '날짜','Lt_before_posing', 'Lt_after_posing', 'End feel_end_feel_type', 'End feel_comment', '치료계획_plat_tracking', '치료계획_plat_extra_treat', '치료계획_next_extra_treat', '치료계획_next_evaluate', '치료계획_next_schedule' ]]

,환자번호,날짜,Lt_before_posing,Lt_after_posing,End feel_end_feel_type,End feel_comment,치료계획_plat_tracking,치료계획_plat_extra_treat,치료계획_next_extra_treat,치료계획_next_evaluate,치료계획_next_schedule
15191,2212-114,2022-12-23,1.1,1.61,Soft,부드러운 엔드필,"당일 내원, 진료의뢰서 지참하셔서 스캔완료/현",,,,0.0
15192,2212-114,2023-02-18,1.1,1.61,Soft,부드러운 엔드필,"-> 아이 아빠랑 상의해보고 다음번에 정해서 올게요, 보톡스는 위험해서 좀 그렇다고...",,,,0.0
15193,2212-114,2023-03-20,1.1,1.61,Soft,부드러운 엔드필,,물리치료,,증상 ck,30.0
15194,2212-114,2023-04-06,1.1,1.61,Soft,부드러운 엔드필,,물리치료,,증상 ck,14.0
15195,2212-114,2023-04-21,1.1,1.37,Soft,부드러운 엔드필,NaN,NaN,NaN,NaN,NaN
15196,2212-114,2023-05-10,1.1,1.37,Soft,부드러운 엔드필,,물리치료,APS del,,14.0
